In [10]:
import os
os.environ['HADOOP_HOME'] = 'C:\\hadoop'
os.environ['PATH'] = os.environ['PATH'] + ';C:\\hadoop\\bin'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0,io.delta:delta-core_2.12:2.4.0 pyspark-shell'

import subprocess
subprocess.run(['pip', 'install', 'delta-spark==2.4.0'], check=True)
print('Done')

Done


In [11]:
import subprocess
subprocess.run(['pip', 'install', 'delta-spark==2.4.0'], check=True)
print('Done')

Done


In [12]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0,io.delta:delta-core_2.12:2.4.0 pyspark-shell'

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .master('local[*]')
    .appName('EnergyFlow-Bronze')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')
print('Spark session ready')

Spark version: 3.4.0
Spark session ready


In [17]:
raw_stream = (
    spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers', 'localhost:9092')
    .option('subscribe', 'energy.prices')
    .option('startingOffsets', 'earliest')
    .load()
)

print('Schema of raw Kafka stream:')
raw_stream.printSchema()

Schema of raw Kafka stream:
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [14]:
from pyspark.sql.functions import col, current_timestamp

bronze_stream = raw_stream.select(
    col('value').cast('string').alias('raw_json'),
    col('key').cast('string').alias('kafka_key'),
    col('topic').alias('kafka_topic'),
    col('partition').alias('kafka_partition'),
    col('offset').alias('kafka_offset'),
    col('timestamp').alias('kafka_timestamp'),
    current_timestamp().alias('ingested_at')
)

print('Bronze stream schema:')
bronze_stream.printSchema()

Bronze stream schema:
root
 |-- raw_json: string (nullable = true)
 |-- kafka_key: string (nullable = true)
 |-- kafka_topic: string (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- ingested_at: timestamp (nullable = false)



In [18]:
BRONZE_PATH = 'C:/Users/cheth/Energyflow/spark/delta/bronze'
CHECKPOINT_PATH = 'C:/Users/cheth/Energyflow/spark/checkpoints/bronze'

bronze_query = (
    bronze_stream.writeStream
    .format('delta')
    .outputMode('append')
    .option('checkpointLocation', CHECKPOINT_PATH)
    .trigger(processingTime='10 seconds')
    .start(BRONZE_PATH)
)

print('Streaming query started!')
print(f'Status: {bronze_query.status}')
print()
print('Data is flowing to:', BRONZE_PATH)
print('Open http://localhost:4040 to see Spark job progress')

Streaming query started!
Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}

Data is flowing to: C:/Users/cheth/Energyflow/spark/delta/bronze
Open http://localhost:4040 to see Spark job progress


In [19]:
import time
time.sleep(30)
print(bronze_query.status)
print(bronze_query.lastProgress)

{'message': "Terminated with exception: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'", 'isDataAvailable': False, 'isTriggerActive': False}
None
